# Paper Hit Probability — citation percentile within FoR division × year cohorts

Each publication's **citation percentile** within its **field × publication-year** cohort, per
window (C3 / C5 / C10 / C_all). 0.99 = top 1 %, 0.95 = top 5 %, 0.90 = top 10 %. Primary key:
`paper_id`. Same percentile as `OpenAlex/notebook/paper_hit_probability.ipynb`:
`rank(pct=True, method='min')` within `(FoS, year)`.

## Input
```
Dimensions/output/paper_citation.parquet   # paper_id, C_3, C_5, C_10, C_all
Dimensions/output/paper_metadata.parquet   # paper_id, year, FoS_0, FoS_rep
```
Cohort **FoS** = `FoS_rep` (the first ANZSRC FoR 2020 division listed) when present, else the
first entry of `FoS_0`. The label set is the 23 FoR divisions, not OpenAlex's 26 fields — the
metric is unchanged, the partition it is computed within is not.

## Output
`Dimensions/output/paper_hit_probability.parquet` — `paper_id, FoS, year, pctl_c3, pctl_c5, pctl_c10, pctl_call`.

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
CIT    = f'{OUT}/paper_citation.parquet'
META   = f'{OUT}/paper_metadata.parquet'
OUT_FP = f'{OUT}/paper_hit_probability.parquet'
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph         19.00 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr          not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal        1.24 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos            0.77 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

In [2]:
WINS = [('c3', 'C_3'), ('c5', 'C_5'), ('c10', 'C_10'), ('call', 'C_all')]
def add_percentile(df, cohort_cols):
    """Within each cohort partition, the citation PERCENTILE per window in [0, 1]
    (higher = more cited). rank(ascending=True, method='min') puts zero-citation ties near 0 and the
    top near 1; e.g. top 1% / 5% / 10% correspond to pctl >= 0.99 / 0.95 / 0.90."""
    for wtag, wcol in WINS:
        df[f'pctl_{wtag}'] = df.groupby(cohort_cols)[wcol].rank(pct=True, ascending=True, method='min').astype('float32')
    return df

## 1. Join citations with FoS + year (pyarrow, then one field at a time)

In [3]:
%%time
# 1. Join citations with FoS + year (pyarrow, then one field at a time)
cit = pq.read_table(CIT, columns=['paper_id', 'C_3', 'C_5', 'C_10', 'C_all'])
met = pq.read_table(META, columns=['paper_id', 'FoS_rep', 'FoS_0', 'year'])
print(f'citations {cit.num_rows:,} | metadata {met.num_rows:,}')
tbl = cit.join(met, keys='paper_id', join_type='inner')
del cit, met; gc.collect()
df = tbl.to_pandas(); del tbl; gc.collect()

# primary FoS = FoS_rep, else the first element of FoS_0 — as before
df['FoS'] = df['FoS_rep'].fillna(df['FoS_0'].str.split(';').str[0])
df = df.dropna(subset=['FoS', 'year'])
df['year'] = df['year'].astype(int)
for c in ['C_3', 'C_5', 'C_10', 'C_all']:
    df[c] = df[c].fillna(0).astype('int64')
df = df[['paper_id', 'FoS', 'year', 'C_3', 'C_5', 'C_10', 'C_all']]
gc.collect()
print(f'papers with FoS + year: {len(df):,} across {df.FoS.nunique()} fields')

citations 155,441,856 | metadata 155,507,994
papers with FoS + year: 119,043,976 across 22 fields


## 2. Compute percentiles + save

In [4]:
%%time
# 2. Percentiles within (FoS, year), one field at a time, then save
outs = []
for i, (fos, g) in enumerate(df.groupby('FoS', sort=False)):
    g = add_percentile(g.copy(), ['FoS', 'year'])
    outs.append(g[['paper_id', 'FoS', 'year'] + [f'pctl_{w}' for w in ['c3','c5','c10','call']]])
    if (i + 1) % 5 == 0:
        print(f'  {i+1} fields done', flush=True)
out = pd.concat(outs, ignore_index=True); del outs; gc.collect()
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows, {len(out.columns)} cols)')
print('citation percentile summary:')
print(out[[f'pctl_{w}' for w in ['c3','c5','c10','call']]].describe().round(4).to_string())
display(out.head(8))

  5 fields done
  10 fields done
  15 fields done
  20 fields done
WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_hit_probability.parquet  (119,043,976 rows, 7 cols)
citation percentile summary:
            pctl_c3       pctl_c5      pctl_c10     pctl_call
count  1.190440e+08  1.190440e+08  1.190440e+08  1.190440e+08
mean   3.554000e-01  3.757000e-01  3.921000e-01  4.078000e-01
std    3.754000e-01  3.754000e-01  3.754000e-01  3.754000e-01
min    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
50%    3.342000e-01  3.770000e-01  4.054000e-01  4.327000e-01
75%    7.069000e-01  7.176000e-01  7.243000e-01  7.323000e-01
max    1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00


,paper_id,FoS,year,pctl_c3,pctl_c5,pctl_c10,pctl_call
0,pub.1000278773,Biomedical and Clinical Sciences,2007,0.405837,0.449765,0.630513,0.634127
1,pub.1000278774,Biomedical and Clinical Sciences,2008,0.404194,0.360628,0.508015,0.530698
2,pub.1000278777,Biomedical and Clinical Sciences,1919,0.000071,0.000071,0.849343,0.881292
3,pub.1000278791,Biomedical and Clinical Sciences,2004,0.641879,0.600804,0.624551,0.557103
4,pub.1000278795,Biomedical and Clinical Sciences,2013,0.000001,0.000001,0.377209,0.416294
5,pub.1000278798,Biomedical and Clinical Sciences,1979,0.000005,0.000005,0.000005,0.000005
6,pub.1000278800,Biomedical and Clinical Sciences,1997,0.893984,0.904646,0.884558,0.851708
7,pub.1000278802,Biomedical and Clinical Sciences,2014,0.953193,0.936439,0.923788,0.923725
